# Explainable Cardiovascular Disease Risk Prediction

This Kaggle notebook is a self-contained version of the hackathon project. It downloads a public Cleveland Heart Disease dataset directly, performs exploratory analysis, trains multiple machine learning models, compares their performance, and explains the best tree model with SHAP.

The local project also supports the staged `heart_processed.csv`, `cardio_base.csv`, and ECG files. This notebook uses an inline public download so it can run on Kaggle without local files.

In [ ]:
# Kaggle normally includes most packages, but this keeps the notebook portable.
import importlib.util
import sys
import subprocess

for package in ['xgboost', 'shap', 'imblearn']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

## 1. Imports And Data Loading

The Cleveland dataset uses `?` for missing values and a multi-class target. We convert the target to binary: `0 = no disease`, `1 = disease`.

In [ ]:
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 100)

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
urlretrieve(url, 'processed.cleveland.data')

df = pd.read_csv('processed.cleveland.data', names=columns, na_values='?')
df['target'] = (df['target'] > 0).astype(int)
print(df.shape)
display(df.head())

## 2. EDA Summary

In [ ]:
display(pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(),
}))

display(df.describe().T)
print('Target distribution:')
display(df['target'].value_counts().rename_axis('target').reset_index(name='count'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.countplot(data=df, x='target', ax=axes[0, 0])
axes[0, 0].set_title('Class Distribution')

sns.histplot(data=df, x='age', hue='target', kde=True, bins=20, ax=axes[0, 1])
axes[0, 1].set_title('Age by Target')

sns.boxplot(data=df, x='target', y='trestbps', ax=axes[1, 0])
axes[1, 0].set_title('Resting BP by Target')

sns.boxplot(data=df, x='target', y='chol', ax=axes[1, 1])
axes[1, 1].set_title('Cholesterol by Target')

plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(numeric_only=True), cmap='vlag', center=0, annot=False)
plt.title('Feature Correlation Heatmap')
plt.show()

## 3. Preprocessing

Numeric features are median-imputed and standardized. Categorical features are mode-imputed and one-hot encoded. SMOTE is applied only to the training split.

In [ ]:
X = df.drop(columns='target')
y = df['target']

categorical = ['cp', 'restecg', 'slope', 'thal']
numerical = [col for col in X.columns if col not in categorical]

try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numerical),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', encoder)]), categorical),
    ],
    verbose_feature_names_out=False,
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out().tolist()

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE:', pd.Series(y_train_smote).value_counts().to_dict())
print('Feature count:', len(feature_names))

## 4. Train And Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.9, colsample_bytree=0.9, objective='binary:logistic', eval_metric='logloss', random_state=42, n_jobs=-1),
    'SVM RBF': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
}

rows = []
roc_data = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'auc_roc': 'roc_auc'}

for name, model in models.items():
    cv_scores = cross_validate(model, X_train_smote, y_train_smote, cv=cv, scoring=scoring, n_jobs=-1)
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_data[name] = (fpr, tpr, roc_auc_score(y_test, y_score))
    rows.append({
        'model': name,
        'cv_auc_roc': cv_scores['test_auc_roc'].mean(),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_test, y_score),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
    })

leaderboard = pd.DataFrame(rows).sort_values(['auc_roc', 'f1'], ascending=False).reset_index(drop=True)
leaderboard.insert(0, 'rank', range(1, len(leaderboard) + 1))
display(leaderboard)

In [ ]:
plot_df = leaderboard.melt(id_vars='model', value_vars=['auc_roc', 'f1'], var_name='metric', value_name='score')
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x='model', y='score', hue='metric')
plt.ylim(0, 1)
plt.title('Model Comparison: AUC-ROC and F1')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for name, (fpr, tpr, model_auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f'{name} (AUC={model_auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. SHAP Analysis

The best tree model, XGBoost, is used for SHAP explanations because tree-based SHAP is efficient and produces patient-level explanations suitable for a clinical audience.

In [ ]:
xgb_model = models['XGBoost']
X_test_df = pd.DataFrame(X_test, columns=feature_names)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_df)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(shap_values, X_test_df, feature_names=feature_names, show=False)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, X_test_df, feature_names=feature_names, plot_type='bar', show=False)
plt.title('Mean Absolute SHAP Importance')
plt.tight_layout()
plt.show()

importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)
display(importance.head(10))

In [ ]:
y_score = xgb_model.predict_proba(X_test_df)[:, 1]
y_pred = (y_score >= 0.5).astype(int)
base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[1]

explanation = shap.Explanation(
    values=shap_values,
    base_values=np.repeat(base_value, X_test_df.shape[0]),
    data=X_test_df.values,
    feature_names=feature_names,
)

def first_index(mask):
    matches = np.flatnonzero(mask)
    return int(matches[0]) if len(matches) else None

cases = {
    'True positive': first_index((y_test.values == 1) & (y_pred == 1)),
    'True negative': first_index((y_test.values == 0) & (y_pred == 0)),
    'False positive': first_index((y_test.values == 0) & (y_pred == 1)),
}

for label, idx in cases.items():
    if idx is None:
        print(f'{label}: unavailable in this split')
        continue
    print(f'{label}: index={idx}, probability={y_score[idx]:.3f}, true={y_test.iloc[idx]}, pred={y_pred[idx]}')
    shap.plots.waterfall(explanation[idx], max_display=10, show=False)
    plt.title(label)
    plt.tight_layout()
    plt.show()

In [ ]:
for feature in importance['feature'].head(3):
    shap.dependence_plot(feature, shap_values, X_test_df, feature_names=feature_names, show=False)
    plt.title(f'SHAP Dependence: {feature}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrates an end-to-end explainable CVD risk prediction workflow using public data. The main contribution is not only comparing multiple models, but also adding SHAP-based global and patient-level explanations so risk predictions can be inspected rather than treated as black boxes.

### Novel Contributions

- Multiple model families are trained and compared under the same preprocessing protocol.
- SMOTE is applied only to training data to reduce class imbalance leakage.
- SHAP identifies global risk drivers and explains individual predictions with waterfall plots.
- The workflow maps directly to a Streamlit demo for live risk prediction.

### Limitations

- The Cleveland dataset is small, so metrics may vary across splits.
- SHAP explains model behavior, not causal medical relationships.
- This is educational software and is not intended for diagnosis or treatment decisions.
- Future work should validate on larger, more diverse cohorts and include calibration analysis.